# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/Users/t.yan/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [7]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

X = df.drop(columns=["Class"]).values
y = df["Class"].values


In [8]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [9]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

y_train_categorical = to_categorical(y_train, num_classes=num_classes)
y_test_categorical = to_categorical(y_test, num_classes=num_classes)


In [11]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.
model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])


In [12]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_scaled,
    y_train_categorical,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
13/13 [==============================] - 0s 7ms/step - loss: 0.9489 - accuracy: 0.5556 - val_loss: 0.8134 - val_accuracy: 0.7200
Epoch 2/20
13/13 [==============================] - 0s 2ms/step - loss: 0.6900 - accuracy: 0.8485 - val_loss: 0.5804 - val_accuracy: 1.0000
Epoch 3/20
13/13 [==============================] - 0s 2ms/step - loss: 0.4937 - accuracy: 0.9293 - val_loss: 0.4003 - val_accuracy: 0.9600
Epoch 4/20
13/13 [==============================] - 0s 2ms/step - loss: 0.3426 - accuracy: 0.9798 - val_loss: 0.2652 - val_accuracy: 0.9600
Epoch 5/20
13/13 [==============================] - 0s 2ms/step - loss: 0.2371 - accuracy: 0.9798 - val_loss: 0.1738 - val_accuracy: 0.9600
Epoch 6/20
13/13 [==============================] - 0s 2ms/step - loss: 0.1647 - accuracy: 0.9798 - val_loss: 0.1228 - val_accuracy: 0.9600
Epoch 7/20
13/13 [==============================] - 0s 2ms/step - loss: 0.1198 - accuracy: 0.9899 - val_loss: 0.0945 - val_accuracy: 0.9600
Epoch 8/20
13/13 [==

In [13]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

y_pred_probs = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

# Calculate accuracy
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test_categorical, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=wine.target_names))

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

2/2 [==============================] - 0s 1ms/step
Test Accuracy: 0.9815

Classification Report:
              precision    recall  f1-score   support

     class_0       0.95      1.00      0.97        18
     class_1       1.00      0.95      0.98        21
     class_2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix:
[[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]


In [15]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the TFLite model to a file
with open("model_base.tflite", "wb") as f:
    f.write(tflite_model)

import os
file_size = os.path.getsize("model_base.tflite") / 1024
print(f"TFLite model saved as 'model_base.tflite'")
print(f"File size: {file_size:.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmp83e11zoj/assets


INFO:tensorflow:Assets written to: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmp83e11zoj/assets


TFLite model saved as 'model_base.tflite'
File size: 14.07 KB


2026-07-24 21:22:50.791932: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-24 21:22:50.791947: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-24 21:22:50.792083: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmp83e11zoj
2026-07-24 21:22:50.792582: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-24 21:22:50.792586: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmp83e11zoj
2026-07-24 21:22:50.794015: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-24 21:22:50.813531: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmp83e11zoj
2026-07-

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [29]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
        pass

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        
        converter.target_spec.supported_types = [tf.float16]

        pass

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        pass

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    tflite_model = converter.convert()
    

    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    input_scale = input_details[0]['quantization'][0]
    input_zero_point = input_details[0]['quantization'][1]
    is_input_quantized = input_scale != 0.0
    
    output_scale = output_details[0]['quantization'][0]
    output_zero_point = output_details[0]['quantization'][1]
    is_output_quantized = output_scale != 0.0
    
    # Run inference on all test samples
    y_pred = []
    for i in range(len(X_test)):
        input_data = X_test[i:i+1].astype(np.float32)
  
        if is_input_quantized:
            input_data = input_data / input_scale + input_zero_point
            input_data = input_data.astype(np.int8)
        
        interpreter.set_tensor(input_details[0]['index'], input_data)
        
        interpreter.invoke()
        
        output_data = interpreter.get_tensor(output_details[0]['index'])
        
        if is_output_quantized:
            output_data = (output_data.astype(np.float32) - output_zero_point) * output_scale
        
        y_pred.append(np.argmax(output_data[0]))

    # Convert to numpy array
    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)

    accuracy = np.mean(y_pred == y_true)

    # Step 4: Report results.
    file_size_kb = os.path.getsize(filename) / 1024 
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb:.2f} KB")
    
    # Print classification report and confusion matrix
    print(f"\nClassification Report for {quant_type.upper()} model:")
    print(classification_report(y_true, y_pred, target_names=wine.target_names))
    
    print(f"Confusion Matrix for {quant_type.upper()} model:")
    print(confusion_matrix(y_true, y_pred))

In [30]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

quantize_and_evaluate(model, X_test_scaled, y_test_categorical, 'int8', 'model_int8.tflite')

quantize_and_evaluate(model, X_test_scaled, y_test_categorical, 'float16', 'model_float16.tflite')

quantize_and_evaluate(model, X_test_scaled, y_test_categorical, 'dynamic', 'model_dynamic.tflite')

INFO:tensorflow:Assets written to: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmpo1eouh8q/assets


INFO:tensorflow:Assets written to: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmpo1eouh8q/assets



INT8 TFLite model size: 5.74 KB

Classification Report for INT8 model:
              precision    recall  f1-score   support

     class_0       0.95      1.00      0.97        18
     class_1       1.00      0.95      0.98        21
     class_2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix for INT8 model:
[[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]


/Users/t.yan/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-07-25 01:18:17.452099: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-25 01:18:17.452114: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-25 01:18:17.452293: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmpo1eouh8q
2026-07-25 01:18:17.452809: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-25 01:18:17.452813: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmpo1eouh8q
2026-07-25 01:18:17.454202: I tensorflow/cc/saved_model/loader.cc:233] Re

INFO:tensorflow:Assets written to: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmphu6hi2pk/assets


...

2026-07-25 01:18:17.741679: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-25 01:18:17.741700: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-25 01:18:17.741831: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmphu6hi2pk
2026-07-25 01:18:17.742347: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-25 01:18:17.742352: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmphu6hi2pk
2026-07-25 01:18:17.743588: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-25 01:18:17.763386: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmphu6hi2pk
2026-07-

INFO:tensorflow:Assets written to: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmp8st5b1z9/assets


INFO:tensorflow:Assets written to: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmp8st5b1z9/assets



DYNAMIC TFLite model size: 8.17 KB

Classification Report for DYNAMIC model:
              precision    recall  f1-score   support

     class_0       0.95      1.00      0.97        18
     class_1       1.00      0.95      0.98        21
     class_2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix for DYNAMIC model:
[[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]


2026-07-25 01:18:18.031037: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-25 01:18:18.031054: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-25 01:18:18.031177: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmp8st5b1z9
2026-07-25 01:18:18.031783: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-25 01:18:18.031787: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmp8st5b1z9
2026-07-25 01:18:18.033035: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-25 01:18:18.053209: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmp8st5b1z9
2026-07-

## Problem 1 - Part (c)

### Pruning

In [35]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

dataset_size = X_train_scaled.shape[0]
batch_size = 8
epochs = 20
total_steps = int(dataset_size / batch_size * epochs)

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=total_steps
)

In [39]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()
pruned_model = Sequential([
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(64, activation='relu', input_shape=(num_features,)),
        pruning_schedule=pruning_schedule
    ),
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule
    ),
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(num_classes, activation='softmax'),
        pruning_schedule=pruning_schedule
    )
])

In [40]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2 
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()
]

# Train the pruned model
pruned_history = pruned_model.fit(
    X_train_scaled,
    y_train_categorical,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 1s 8ms/step - loss: 0.8610 - accuracy: 0.5960 - val_loss: 0.5709 - val_accuracy: 0.8800
Epoch 2/10
13/13 [==============================] - 0s 2ms/step - loss: 0.5940 - accuracy: 0.9192 - val_loss: 0.3940 - val_accuracy: 0.9600
Epoch 3/10
13/13 [==============================] - 0s 2ms/step - loss: 0.4109 - accuracy: 0.9697 - val_loss: 0.2861 - val_accuracy: 0.9600
Epoch 4/10
13/13 [==============================] - 0s 2ms/step - loss: 0.2860 - accuracy: 0.9798 - val_loss: 0.2105 - val_accuracy: 0.9600
Epoch 5/10
13/13 [==============================] - 0s 2ms/step - loss: 0.1979 - accuracy: 1.0000 - val_loss: 0.1576 - val_accuracy: 0.9600
Epoch 6/10
13/13 [==============================] - 0s 2ms/step - loss: 0.1412 - accuracy: 1.0000 - val_loss: 0.1332 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 2ms/step - loss: 0.1055 - accuracy: 1.0000 - val_loss: 0.1065 - val_accuracy: 0.9600
Epoch 8/10
13/13 [==

In [41]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.
# Remove the pruning wrappers from the model

stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

# Convert the stripped model to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(stripped_pruned_model)
tflite_pruned_model = converter.convert()

# Save the TFLite model
with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_pruned_model)

# Print the file size in KB
file_size_kb = os.path.getsize("model_pruned.tflite") / 1024
print(f"Pruned TFLite model saved as 'model_pruned.tflite'")
print(f"File size: {file_size_kb:.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmptiy8dkju/assets


INFO:tensorflow:Assets written to: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmptiy8dkju/assets
2026-07-28 17:11:59.821880: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.


Pruned TFLite model saved as 'model_pruned.tflite'
File size: 14.18 KB


2026-07-28 17:11:59.822161: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-28 17:11:59.822657: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmptiy8dkju
2026-07-28 17:11:59.823140: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-28 17:11:59.823146: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmptiy8dkju
2026-07-28 17:11:59.825724: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-28 17:11:59.839107: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmptiy8dkju
2026-07-28 17:11:59.843204: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took

In [42]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

y_pred_stripped = np.argmax(stripped_pruned_model.predict(X_test_scaled), axis=1)

print(f"Accuracy: {np.mean(y_pred_stripped == y_test):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_stripped, target_names=wine.target_names))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_stripped))

2/2 [==============================] - 0s 2ms/step
Accuracy: 1.0000

Classification Report:
               precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      1.00      1.00        21
     class_2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
 [[18  0  0]
 [ 0 21  0]
 [ 0  0 15]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [43]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

student_model = Sequential([
    Dense(32, activation='relu', input_shape=(num_features,)),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

student_model.summary()

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_15 (Dense)            (None, 32)                448       
                                                                 
 dense_16 (Dense)            (None, 16)                528       
                                                                 
 dense_17 (Dense)            (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [44]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

teacher_soft_labels = model.predict(X_train_scaled)

4/4 [==============================] - 0s 990us/step


In [45]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

y_train_combined = np.concatenate([y_train_categorical, teacher_soft_labels], axis=1)

def distillation_loss(y_true_combined, y_pred):

    y_true_hard = y_true_combined[:, :num_classes] 
    y_true_soft = y_true_combined[:, num_classes:] 
    alpha = 0.5
    
    # Compute hard loss (using hard labels)
    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    
    # Compute soft loss (using soft labels)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)
   
    return alpha * hard_loss + (1 - alpha) * soft_loss

In [46]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

# Train for 10 epochs, batch_size=8, validation_split=0.2
student_history = student_model.fit(
    X_train_scaled,
    y_train_combined,  # Use the combined hard+soft labels
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 1s 7ms/step - loss: 1.2286 - accuracy: 0.1919 - val_loss: 1.1173 - val_accuracy: 0.2400
Epoch 2/10
13/13 [==============================] - 0s 2ms/step - loss: 1.0156 - accuracy: 0.4242 - val_loss: 0.9226 - val_accuracy: 0.6000
Epoch 3/10
13/13 [==============================] - 0s 2ms/step - loss: 0.8508 - accuracy: 0.7172 - val_loss: 0.7591 - val_accuracy: 0.8000
Epoch 4/10
13/13 [==============================] - 0s 2ms/step - loss: 0.7039 - accuracy: 0.8586 - val_loss: 0.6251 - val_accuracy: 0.8800
Epoch 5/10
13/13 [==============================] - 0s 2ms/step - loss: 0.5771 - accuracy: 0.9596 - val_loss: 0.4973 - val_accuracy: 0.9600
Epoch 6/10
13/13 [==============================] - 0s 2ms/step - loss: 0.4669 - accuracy: 0.9697 - val_loss: 0.3999 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 2ms/step - loss: 0.3710 - accuracy: 0.9697 - val_loss: 0.3239 - val_accuracy: 0.9600
Epoch 8/10
13/13 [==

In [47]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd_model = converter.convert()

# Save the TFLite model
with open("model_kd.tflite", "wb") as f:
    f.write(tflite_kd_model)

# Print the file size in KB
file_size_kd = os.path.getsize("model_kd.tflite") / 1024
print(f"Knowledge Distillation TFLite model saved as 'model_kd.tflite'")
print(f"File size: {file_size_kd:.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmpo8p2feb2/assets


INFO:tensorflow:Assets written to: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmpo8p2feb2/assets
2026-07-29 15:54:22.864106: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.


Knowledge Distillation TFLite model saved as 'model_kd.tflite'
File size: 6.14 KB


2026-07-29 15:54:22.864460: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-29 15:54:22.864751: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmpo8p2feb2
2026-07-29 15:54:22.865303: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-29 15:54:22.865308: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmpo8p2feb2
2026-07-29 15:54:22.867728: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-29 15:54:22.890856: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmpo8p2feb2
2026-07-29 15:54:22.897037: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took

In [48]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

y_pred_student = np.argmax(student_model.predict(X_test_scaled), axis=1)

print("\nStudent Model (Knowledge Distillation) Classification Report:")
print(classification_report(y_test, y_pred_student, target_names=wine.target_names))

print("Student Model (Knowledge Distillation) Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_student))

2/2 [==============================] - 0s 1ms/step

Student Model (Knowledge Distillation) Classification Report:
              precision    recall  f1-score   support

     class_0       0.90      1.00      0.95        18
     class_1       0.90      0.86      0.88        21
     class_2       0.93      0.87      0.90        15

    accuracy                           0.91        54
   macro avg       0.91      0.91      0.91        54
weighted avg       0.91      0.91      0.91        54

Student Model (Knowledge Distillation) Confusion Matrix:
[[18  0  0]
 [ 2 18  1]
 [ 0  2 13]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [56]:
# code unuseful 
# Step 1: Build an even smaller student model for knowledge distillation
tiny_student = Sequential([
    Dense(16, activation='relu', input_shape=(num_features,)),  # Even smaller
    Dense(8, activation='relu'),
    Dense(num_classes, activation='softmax')
])

# Step 2: Compile with distillation loss
tiny_student.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

# Step 3: Train the tiny student
tiny_history = tiny_student.fit(
    X_train_scaled,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

# Step 4: Apply pruning to the tiny student
pruning_schedule_tiny = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.3,
    final_sparsity=0.6,
    begin_step=0,
    end_step=int(X_train_scaled.shape[0] / 8 * 10)
)

# Wrap layers with pruning
pruned_tiny_model = Sequential([
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(16, activation='relu', input_shape=(num_features,)),
        pruning_schedule=pruning_schedule_tiny
    ),
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(8, activation='relu'),
        pruning_schedule=pruning_schedule_tiny
    ),
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(num_classes, activation='softmax'),
        pruning_schedule=pruning_schedule_tiny
    )
])

# Step 5: Compile the pruned model
pruned_tiny_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

# Step 6: Train the pruned model from scratch (don't transfer weights)
# The pruned model has extra variables, so we train it from scratch
pruned_tiny_history = pruned_tiny_model.fit(
    X_train_scaled,
    y_train_combined,
    epochs=10,  # Train with pruning from the start
    batch_size=8,
    validation_split=0.2,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()],
    verbose=1
)

# Step 7: Strip pruning wrappers
stripped_pruned_tiny = tfmot.sparsity.keras.strip_pruning(pruned_tiny_model)

# Step 8: Convert to TFLite with INT8 quantization
def representative_data_gen():
    for i in range(min(100, len(X_train_scaled))):
        yield [X_train_scaled[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_pruned_tiny)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_ultra_model = converter.convert()

# Step 9: Save and evaluate
with open("model_ultra_compressed.tflite", "wb") as f:
    f.write(tflite_ultra_model)

ultra_size = os.path.getsize("model_ultra_compressed.tflite") / 1024
print(f"Ultra-compressed model size: {ultra_size:.2f} KB")

# Step 10: Evaluate the ultra-compressed model
interpreter = tf.lite.Interpreter(model_content=tflite_ultra_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Run inference
y_pred_ultra = []
for i in range(len(X_test_scaled)):
    # Quantize input
    input_scale, input_zero_point = input_details[0]['quantization']
    input_data = X_test_scaled[i:i+1] / input_scale + input_zero_point
    input_data = input_data.astype(np.int8)
    
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    
    output_data = interpreter.get_tensor(output_details[0]['index'])
    
    # Dequantize output
    output_scale, output_zero_point = output_details[0]['quantization']
    output_data = (output_data.astype(np.float32) - output_zero_point) * output_scale
    
    y_pred_ultra.append(np.argmax(output_data[0]))

y_pred_ultra = np.array(y_pred_ultra)
accuracy_ultra = np.mean(y_pred_ultra == y_test)

print(f"\nUltra-compressed model accuracy: {accuracy_ultra:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_ultra, target_names=wine.target_names))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_ultra))

Epoch 1/10
13/13 [==============================] - 0s 6ms/step - loss: 1.0513 - accuracy: 0.3434 - val_loss: 1.1497 - val_accuracy: 0.2800
Epoch 2/10
13/13 [==============================] - 0s 2ms/step - loss: 0.9762 - accuracy: 0.4444 - val_loss: 1.0865 - val_accuracy: 0.3600
Epoch 3/10
13/13 [==============================] - 0s 2ms/step - loss: 0.9127 - accuracy: 0.5960 - val_loss: 1.0306 - val_accuracy: 0.4800
Epoch 4/10
13/13 [==============================] - 0s 2ms/step - loss: 0.8551 - accuracy: 0.6970 - val_loss: 0.9764 - val_accuracy: 0.5200
Epoch 5/10
13/13 [==============================] - 0s 2ms/step - loss: 0.7971 - accuracy: 0.7071 - val_loss: 0.9223 - val_accuracy: 0.5600
Epoch 6/10
13/13 [==============================] - 0s 2ms/step - loss: 0.7432 - accuracy: 0.7273 - val_loss: 0.8703 - val_accuracy: 0.6000
Epoch 7/10
13/13 [==============================] - 0s 2ms/step - loss: 0.6915 - accuracy: 0.7576 - val_loss: 0.8219 - val_accuracy: 0.6400
Epoch 8/10
13/13 [==

INFO:tensorflow:Assets written to: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmpnhqst6l1/assets


Ultra-compressed model size: 3.02 KB

Ultra-compressed model accuracy: 0.6296

Classification Report:
              precision    recall  f1-score   support

     class_0       0.00      0.00      0.00        18
     class_1       0.51      0.90      0.66        21
     class_2       0.88      1.00      0.94        15

    accuracy                           0.63        54
   macro avg       0.47      0.63      0.53        54
weighted avg       0.44      0.63      0.52        54


Confusion Matrix:
[[ 0 18  0]
 [ 0 19  2]
 [ 0  0 15]]


/Users/t.yan/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-07-29 18:59:05.458670: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-29 18:59:05.458682: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-29 18:59:05.458817: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmpnhqst6l1
2026-07-29 18:59:05.459199: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-29 18:59:05.459206: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/3d/q5ysxhdn59d06w5th7xvl18m0000gn/T/tmpnhqst6l1
2026-07-29 18:59:05.459999: I tensorflow/cc/saved_model/loader.cc:233] Re

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
